In [ ]:
x_train = pd.read_csv('x_train.csv')
y_train = pd.read_csv('y_train.csv')
x_test = pd.read_csv('x_test.csv')

#merge
train_df = pd.merge(x_train, y_train, on='video_id')
test_df = x_test.copy()

#advanced help func

def create_text_blob(df):
    #combines all text fields for one lower case string
    return (df['author_bio'].fillna('') + " " +
            df['video_transcription_text'].fillna('') + " " +
            df['sound_name'].fillna('')).str.lower()

def get_region_advanced(row, blob):
    # we try to find region despite missing, using keywords and character set

    # If region is already present, keep it
    if pd.notna(row['user_region']) and row['user_region'] != 'Missing':
        return row['user_region']

    t = blob # temp blob

    # ISRAEL (IL)
    if any(x in t for x in ['tel aviv', 'israel', 'jerusalem', 'shalom', 'idf', 'hummus', 'eilat', 'kibbutz', 'matkot']):
        return 'IL'
    if re.search(r'[\u0590-\u05FF]', t): #hebrew unicode
        return 'IL'

    # JAPAN (JP)
    if any(x in t for x in ['japan', 'tokyo', 'shibuya', 'osaka', 'kyoto', 'anime', 'sushi', 'sakura', 'kawaii', 'hokkaido', 'nintendo', 'samurai', 'ramen']):
        return 'JP'
    if re.search(r'[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff]', t): #japanese and chinese unicode
        return 'JP'

    # BRAZIL (BR)
    if any(x in t for x in ['brazil', 'brasil', 'rio', 'janeiro', 'paulo', 'praia', 'carnival', 'samba', 'flamengo', 'neymar', 'reais']):
        return 'BR'

    # UNITED STATES (US)
    if any(x in t for x in ['usa', 'nyc', 'new york', 'los angeles', 'california', 'miami', 'dollars', 'superbowl', 'nfl', 'nba', 'hollywood', 'chicago', 'texas', 'vegas', 'football', 'freedom', 'liberty']):
        return 'US'

    return 'Missing' # really is missing

def get_tags_advanced(row, blob):
    # if visual_tag is missing, we try to associate it one using a fix of some we picked

    # exists
    if pd.notna(row['visual_tags']) and row['visual_tags'] != 'Missing':
        return row['visual_tags']

    #these 2 are high confidence tags
    sound = row['sound_name']
    if sound == 'Dance Monkey': return 'Bedroom_Dance'
    if sound == 'Funny Laugh 3': return 'Street_Prank'

    #now we try based on words in the blob we created
    t = blob
    if any(x in t for x in ['cook', 'kitchen', 'recipe', 'food']): return 'Cooking_Kitchen'
    if any(x in t for x in ['gym', 'workout', 'fitness', 'muscle']): return 'Gym_Workout'
    if any(x in t for x in ['beach', 'sun', 'ocean', 'sea']): return 'Beach_Sunset'
    if any(x in t for x in ['dance', 'dancing', 'choreography']): return 'Bedroom_Dance'
    if any(x in t for x in ['prank', 'joke', 'funny', 'laugh']): return 'Street_Prank'

    return 'Missing' # really is missing

def get_local_hour(row):
    # we take into considiration not only time, but relation to other time zone, UTC
    timezone_offsets = {
        'US': -5,
        'IL': 2,
        'JP': 9,
        'BR': -3,
        'Missing': 0
    }
    offset = timezone_offsets.get(row['user_region'], 0)

    return (row['hour'] + offset) % 24 # in order to remain in the 24 hour window

def preprocess_pipeline(df):
    df = df.copy()

    #text
    df['text_blob'] = create_text_blob(df)

    #imputation
    df['user_region'] = df.apply(lambda row: get_region_advanced(row, row['text_blob']), axis=1)
    df['visual_tags'] = df.apply(lambda row: get_tags_advanced(row, row['text_blob']), axis=1)

    #text stats
    df['emoji_count'] = df['author_bio'].apply(lambda x: len(re.findall(r'[^\x00-\x7F]+', str(x))))
    df['bio_word_count'] = df['author_bio'].fillna('').apply(lambda x: len(str(x).split()))
    df['bio_len'] = df['author_bio'].str.len().fillna(0)
    df['trans_len'] = df['video_transcription_text'].str.len().fillna(0)
    df['has_bio'] = df['author_bio'].notna().astype(int)

    #temp deatures
    df['creation_timestamp'] = pd.to_datetime(df['creation_timestamp'])
    df['hour'] = df['creation_timestamp'].dt.hour
    df['local_hour'] = df.apply(get_local_hour, axis=1) # NEW Feature
    df['day_of_week'] = df['creation_timestamp'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    return df

#processing
train_processed = preprocess_pipeline(train_df)
test_processed = preprocess_pipeline(test_df)

#feature engineering & encoding

#numerical - fills using median
num_cols_to_fill = ['followers_at_upload', 'video_duration_sec']
imputer = SimpleImputer(strategy='median')
train_processed[num_cols_to_fill] = imputer.fit_transform(train_processed[num_cols_to_fill])
test_processed[num_cols_to_fill] = imputer.transform(test_processed[num_cols_to_fill])

#log transform - followers are skewd
train_processed['followers_log'] = np.log1p(train_processed['followers_at_upload'])
test_processed['followers_log'] = np.log1p(test_processed['followers_at_upload'])
train_processed['duration_clean'] = train_processed['video_duration_sec']
test_processed['duration_clean'] = test_processed['video_duration_sec']

#replaces it with the average virality score of all videos that used that sound in the training set
sound_means = train_processed.groupby('sound_name')['virality_score'].mean()
global_mean = train_processed['virality_score'].mean()

train_processed['sound_target_enc'] = train_processed['sound_name'].map(sound_means)
test_processed['sound_target_enc'] = test_processed['sound_name'].map(sound_means).fillna(global_mean)

#one-hot , to convert category to binary
cat_cols = ['sound_name', 'visual_tags', 'user_region']
train_dummies = pd.get_dummies(train_processed[cat_cols], drop_first=True)
test_dummies = pd.get_dummies(test_processed[cat_cols], drop_first=True)

#align train and test to have same columns
train_dummies, test_dummies = train_dummies.align(test_dummies, join='left', axis=1, fill_value=0)

#final feature we chose
num_features = [
    'followers_log', 'duration_clean', 'bio_len', 'trans_len',
    'hour', 'local_hour', 'day_of_week', 'is_weekend', 'emoji_count',
    'bio_word_count', 'has_bio', 'sound_target_enc'
]

X = pd.concat([train_processed[num_features], train_dummies], axis=1)
y = train_processed['virality_score']
X_test_final = pd.concat([test_processed[num_features], test_dummies], axis=1)

#scaling so we have mean 0 and s.d. of 1
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_final), columns=X_test_final.columns)


# ccp_alpha=0.05 allows the tree to capture more structure than 0.07
# max_depth=15 prevents overfitting deep branches
dt_model = DecisionTreeRegressor(
    random_state=42,
    min_samples_leaf=10,
    max_depth=15,
    ccp_alpha=0.05,
    min_samples_split=20
)

# cv check
cv_scores = cross_val_score(dt_model, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
print(f"Improved CV MSE: {-cv_scores.mean():.4f}")

#train
dt_model.fit(X_scaled, y)

#testing
y_pred_final = dt_model.predict(X_test_scaled)

# saving to csv
submission = pd.DataFrame({
    'video_id': x_test['video_id'],
    'virality_score': y_pred_final
})

submission.to_csv('submission.csv', index=False)
print("Submission file saved successfully.")